# 🚀 Vision Mamba — Detección de Placas Ecuatorianas
**TrafficVision · MMDetection | plates-ecuadorian v1**

Este notebook realiza:
1. Instalación del entorno MMDetection + Vision Mamba
2. Conversión automática del dataset YOLO → COCO JSON
3. Entrenamiento con backbone Vim-Tiny preentrenado en ImageNet-1k
4. Exportación de métricas por época a `result.csv`
5. Evaluación final en test set con métricas COCO completas

## 1. 📦 Instalación de Dependencias

In [ ]:
import torch, sys
print(f'Python: {sys.version}')
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')

print('\n' + '='*60)
print('⚠️  INSTALACIÓN DESDE CÓDIGO FUENTE (TOMA ~20-30 MINUTOS) ⚠️')
print('El entorno actual de Colab (Python 3.12 + PyTorch 2.11) no tiene')
print('ruedas precompiladas de MMCV. Se compilará desde cero en C++.')
print('Verás mucho texto pasar, es normal. ¡Ve por un café!')
print('='*60 + '\n')

# 1. Instalar dependencias base seguras para Python 3.12
!pip install setuptools packaging ninja -q
!pip install mmengine -q

# 2. Instalar MMCV desde codigo fuente (con verbosidad para ver progreso)
print('\n[1/4] Compilando MMCV... (Esto tomará entre 15 y 20 minutos)')
!pip install mmcv==2.2.0 -v

# 3. Clonar e instalar MMDetection
print('\n[2/4] Instalando MMDetection...')
import os
if not os.path.exists('/content/mmdetection'):
    !git clone https://github.com/open-mmlab/mmdetection.git /content/mmdetection
%cd /content/mmdetection
!pip install -e . -q

# 4. Instalar dependencias de Vision Mamba (lento)
print('\n[3/4] Compilando causal-conv1d... (Tomará unos minutos)')
!pip install causal-conv1d>=1.2.0 --no-build-isolation -v

print('\n[4/4] Compilando mamba-ssm... (Tomará unos minutos)')
!pip install mamba-ssm --no-build-isolation -v

print('\n✅ Todas las dependencias instaladas con éxito. ¡Continúa con la Celda 2!')

In [ ]:
import torch, sys
print(f'Python: {sys.version}')
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')

if '2.1.0' not in torch.__version__:
    raise RuntimeError('⚠️ No reiniciaste el entorno. Por favor reinicia el entorno y vuelve a correr esta celda.')

# ── 2. Instalar MMEngine y MMCV directamente sin openmim para evitar error pkgutil ──
!pip install mmengine -q
!pip install mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch210/index.html -q

# ── 3. MMDetection ──
import os
if not os.path.exists('mmdetection'):
    !git clone https://github.com/open-mmlab/mmdetection.git
%cd mmdetection
!pip install -e . -q

# ── 4. Mamba SSM ──
!pip install causal-conv1d>=1.2.0 --no-build-isolation -q
!pip install mamba-ssm --no-build-isolation -q

print('\n✅ Todas las dependencias instaladas.')

## 2. 📂 Configuración de Rutas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ⚙️  Ajusta estas rutas a tu Google Drive
DATASET_YOLO_ROOT = '/content/drive/MyDrive/TrafficVision/plates ecuadorian'
WORK_DIR          = '/content/work_dirs/vision_mamba_plates'
RESULTS_DIR       = '/content/drive/MyDrive/TrafficVision/ml/training'
CLASS_NAMES       = ['license plate']
MAX_EPOCHS        = 30
BATCH_SIZE        = 4
IMG_SIZE          = 640

import os
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"✅ Dataset YOLO  : {DATASET_YOLO_ROOT}")
print(f"   Work dir      : {WORK_DIR}")
print(f"   Resultados CSV: {RESULTS_DIR}")

## 3. 📥 Descarga del Dataset (Roboflow API → COCO-MMDetection)

Descarga directamente en Colab en formato COCO-MMDetection, listo para MMDetection sin conversión adicional.

In [ ]:
!pip install roboflow -q

from roboflow import Roboflow
import os

%cd /content

rf      = Roboflow(api_key='LPg6zahR6BvuUpVhtsiC')
project = rf.workspace('stevens-workspace-unaqf').project('plates-ecuadorian')
version = project.version(4)
dataset = version.download('coco-mmdetection')

# Roboflow descarga en /content/<project>-<version>/
COCO_DATA_ROOT = dataset.location
print(f'\n✅ Dataset descargado en: {COCO_DATA_ROOT}')

# Verificar estructura descargada
import json
print()
for split in ['train', 'valid', 'test']:
    json_path = os.path.join(COCO_DATA_ROOT, split, '_annotations.coco.json')
    img_dir   = os.path.join(COCO_DATA_ROOT, split, 'images') \
                if os.path.exists(os.path.join(COCO_DATA_ROOT, split, 'images')) \
                else os.path.join(COCO_DATA_ROOT, split)
    if os.path.exists(json_path):
        d = json.load(open(json_path))
        imgs = len(d['images']); anns = len(d['annotations'])
        print(f'  ✅ {split:5s}: {imgs:4d} imgs, {anns:5d} anotaciones')
    else:
        print(f'  ⚠️  {split}: _annotations.coco.json no encontrado en {json_path}')

## 4. 🏗️ Registro del Backbone Vision Mamba

In [ ]:
import sys
sys.path.insert(0, '/content/mmdetection')

backbone_code = """
import torch, torch.nn as nn, math
from functools import partial
from mmdet.registry import MODELS
from mmengine.model import BaseModule

try:
    from mamba_ssm.modules.mamba_simple import Mamba
    from mamba_ssm.ops.triton.layernorm import RMSNorm, rms_norm_fn, layer_norm_fn
    HAS_MAMBA = True
except ImportError:
    HAS_MAMBA = False

def _init_weights(m, n_layer):
    if isinstance(m, nn.Linear) and m.bias is not None:
        if not getattr(m.bias, '_no_reinit', False):
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Embedding):
        nn.init.normal_(m.weight, std=0.02)
    for name, p in m.named_parameters():
        if name in ['out_proj.weight','fc2.weight']:
            nn.init.kaiming_uniform_(p, a=math.sqrt(5))
            with torch.no_grad(): p /= math.sqrt(n_layer)

class Block(nn.Module):
    def __init__(self, dim, mixer_cls, norm_cls, fused, fp32):
        super().__init__()
        self.fused = fused; self.fp32 = fp32
        self.mixer = mixer_cls(dim)
        self.norm  = norm_cls(dim)
    def forward(self, x, residual=None, inference_params=None):
        if not self.fused:
            residual = (x + residual) if residual is not None else x
            x = self.norm(residual.to(self.norm.weight.dtype))
            if self.fp32: residual = residual.float()
        else:
            fn = rms_norm_fn if isinstance(self.norm, RMSNorm) else layer_norm_fn
            x, residual = fn(x, self.norm.weight, self.norm.bias,
                             residual=residual, prenorm=True,
                             residual_in_fp32=self.fp32, eps=self.norm.eps)
        x = self.mixer(x, inference_params=inference_params)
        return x, residual

@MODELS.register_module()
class VisionMamba(BaseModule):
    def __init__(self, img_size=224, patch_size=16, in_chans=3,
                 embed_dim=192, depth=24, out_indices=(5,11,17,23),
                 rms_norm=True, residual_in_fp32=True, fused_add_norm=True,
                 if_abs_pos_embed=True, bimamba_type='v2',
                 final_pool_type='none', if_rope=False, if_rope_residual=False,
                 init_cfg=None):
        super().__init__(init_cfg=init_cfg)
        assert HAS_MAMBA, 'Instala mamba-ssm'
        self.embed_dim = embed_dim; self.depth = depth
        self.out_indices = out_indices; self.if_abs_pos_embed = if_abs_pos_embed
        self.patch_embed = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        num_patches = (img_size // patch_size) ** 2
        if if_abs_pos_embed:
            self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
            nn.init.trunc_normal_(self.pos_embed, std=0.02)
        norm_cls = partial(RMSNorm, eps=1e-5) if (rms_norm and HAS_MAMBA) else partial(nn.LayerNorm, eps=1e-5)
        self.layers = nn.ModuleList([
            Block(embed_dim,
                  partial(Mamba, layer_idx=i, d_state=16, d_conv=4, expand=2),
                  norm_cls, fused_add_norm, residual_in_fp32)
            for i in range(depth)
        ])
        self.out_norms = nn.ModuleList([norm_cls(embed_dim) for _ in out_indices])
        self.apply(partial(_init_weights, n_layer=depth))

    def forward(self, x):
        B, C, H, W = x.shape
        x  = self.patch_embed(x)
        hp, wp = x.shape[2], x.shape[3]
        x  = x.flatten(2).transpose(1, 2)
        if self.if_abs_pos_embed:
            if x.shape[1] != self.pos_embed.shape[1]:
                pe = self.pos_embed.transpose(1,2).reshape(
                    1, self.embed_dim,
                    int(self.pos_embed.shape[1]**0.5),
                    int(self.pos_embed.shape[1]**0.5))
                pe = nn.functional.interpolate(pe, (hp,wp), mode='bicubic', align_corners=False)
                pe = pe.flatten(2).transpose(1,2)
            else:
                pe = self.pos_embed
            x = x + pe
        outs, residual, oi = [], None, 0
        for i, layer in enumerate(self.layers):
            x, residual = layer(x, residual)
            if i in self.out_indices:
                feat = self.out_norms[oi](x).transpose(1,2).reshape(B, self.embed_dim, hp, wp)
                outs.append(feat); oi += 1
        return tuple(outs)
"""

bkb_path = '/content/mmdetection/mmdet/models/backbones/vision_mamba.py'
with open(bkb_path, 'w') as f: f.write(backbone_code)

init_path = '/content/mmdetection/mmdet/models/backbones/__init__.py'
content   = open(init_path).read()
if 'VisionMamba' not in content:
    content = 'from .vision_mamba import VisionMamba\n' + content
    content = content.replace("__all__ = [", "__all__ = [\n    'VisionMamba',")
    open(init_path, 'w').write(content)
    print("✅ VisionMamba registrado.")
else:
    print("ℹ️  VisionMamba ya registrado.")

## 5. ⬇️ Backbone Preentrenado (Vim-Tiny ImageNet-1k)

In [ ]:
import os, urllib.request
os.makedirs('/content/mmdetection/checkpoints', exist_ok=True)
URL  = 'https://huggingface.co/hustvl/Vim-tiny/resolve/main/vim_t_midclstok_ft_in1k_81p3.pth'
CKPT = '/content/mmdetection/checkpoints/vim_tiny_backbone.pth'
if not os.path.exists(CKPT):
    print('⬇️  Descargando Vim-Tiny...')
    urllib.request.urlretrieve(URL, CKPT)
    print('✅ Descarga completada.')
else:
    print(f"✅ Ya existe ({os.path.getsize(CKPT)/(1024**2):.1f} MB)")

## 6. ⚙️ Archivo de Configuración MMDetection

In [ ]:
cfg = f"""
_base_ = ['./configs/_base_/default_runtime.py']

model = dict(
    type='FasterRCNN',
    data_preprocessor=dict(type='DetDataPreprocessor',
        mean=[123.675,116.28,103.53], std=[58.395,57.12,57.375],
        bgr_to_rgb=True, pad_size_divisor=32),
    backbone=dict(type='VisionMamba',
        img_size=224, patch_size=16, embed_dim=192, depth=24,
        out_indices=(5,11,17,23), rms_norm=True, residual_in_fp32=True,
        fused_add_norm=True, if_abs_pos_embed=True, bimamba_type='v2',
        init_cfg=dict(type='Pretrained', checkpoint='checkpoints/vim_tiny_backbone.pth')),
    neck=dict(type='FPN', in_channels=[192,192,192,192], out_channels=256, num_outs=5),
    rpn_head=dict(type='RPNHead', in_channels=256, feat_channels=256,
        anchor_generator=dict(type='AnchorGenerator', scales=[8],
            ratios=[0.3,0.5,1.0,2.0,3.5], strides=[4,8,16,32,64]),
        bbox_coder=dict(type='DeltaXYWHBBoxCoder',
            target_means=[0.,0.,0.,0.], target_stds=[1.,1.,1.,1.]),
        loss_cls=dict(type='CrossEntropyLoss', use_sigmoid=True, loss_weight=1.0),
        loss_bbox=dict(type='L1Loss', loss_weight=1.0)),
    roi_head=dict(type='StandardRoIHead',
        bbox_roi_extractor=dict(type='SingleRoIExtractor',
            roi_layer=dict(type='RoIAlign', output_size=7, sampling_ratio=0),
            out_channels=256, featmap_strides=[4,8,16,32]),
        bbox_head=dict(type='Shared2FCBBoxHead',
            in_channels=256, fc_out_channels=1024, roi_feat_size=7, num_classes=1,
            bbox_coder=dict(type='DeltaXYWHBBoxCoder',
                target_means=[0.,0.,0.,0.], target_stds=[0.1,0.1,0.2,0.2]),
            reg_class_agnostic=False,
            loss_cls=dict(type='CrossEntropyLoss', use_sigmoid=False, loss_weight=1.0),
            loss_bbox=dict(type='GIoULoss', loss_weight=1.0))),
    train_cfg=dict(
        rpn=dict(assigner=dict(type='MaxIoUAssigner',pos_iou_thr=0.7,neg_iou_thr=0.3,
                    min_pos_iou=0.3,match_low_quality=True,ignore_iof_thr=-1),
                 sampler=dict(type='RandomSampler',num=256,pos_fraction=0.5,
                    neg_pos_ub=-1,add_gt_as_proposals=False),
                 allowed_border=-1, pos_weight=-1, debug=False),
        rpn_proposal=dict(nms_pre=2000,max_per_img=1000,
                          nms=dict(type='nms',iou_threshold=0.7),min_bbox_size=0),
        roi=dict(assigner=dict(type='MaxIoUAssigner',pos_iou_thr=0.5,neg_iou_thr=0.5,
                    min_pos_iou=0.5,match_low_quality=False,ignore_iof_thr=-1),
                 sampler=dict(type='RandomSampler',num=512,pos_fraction=0.25,
                    neg_pos_ub=-1,add_gt_as_proposals=True),
                 pos_weight=-1, debug=False)),
    test_cfg=dict(
        rpn=dict(nms_pre=1000,max_per_img=1000,
                 nms=dict(type='nms',iou_threshold=0.7),min_bbox_size=0),
        roi=dict(score_thr=0.05,nms=dict(type='nms',iou_threshold=0.5),max_per_img=100)))

dataset_type  = 'CocoDataset'
data_root     = COCO_DATA_ROOT + '/'
metainfo      = dict(classes=('license plate',))

train_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='LoadAnnotations', with_bbox=True),
    dict(type='RandomFlip', prob=0.5),
    dict(type='RandomChoice', transforms=[
        [dict(type='Resize', scale=({IMG_SIZE},{IMG_SIZE}), keep_ratio=True)],
        [dict(type='Resize', scale=(480,480), keep_ratio=True),
         dict(type='RandomCrop', crop_size=({IMG_SIZE},{IMG_SIZE}), allow_negative_crop=True)]]),
    dict(type='PackDetInputs')]

test_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='Resize', scale=({IMG_SIZE},{IMG_SIZE}), keep_ratio=True),
    dict(type='LoadAnnotations', with_bbox=True),
    dict(type='PackDetInputs',
         meta_keys=('img_id','img_path','ori_shape','img_shape','scale_factor'))]

train_dataloader = dict(batch_size={BATCH_SIZE}, num_workers=2,
    persistent_workers=True,
    sampler=dict(type='DefaultSampler', shuffle=True),
    batch_sampler=dict(type='AspectRatioBatchSampler'),
    dataset=dict(type=dataset_type, metainfo=metainfo,
        data_root=data_root, ann_file='train/_annotations.coco.json',
        data_prefix=dict(img='train/images/'),
        filter_cfg=dict(filter_empty_gt=True, min_size=32),
        pipeline=train_pipeline))

val_dataloader = dict(batch_size=1, num_workers=2,
    persistent_workers=True, drop_last=False,
    sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=dict(type=dataset_type, metainfo=metainfo,
        data_root=data_root, ann_file='valid/_annotations.coco.json',
        data_prefix=dict(img='valid/images/'),
        test_mode=True, pipeline=test_pipeline))

test_dataloader = dict(batch_size=1, num_workers=2,
    persistent_workers=True, drop_last=False,
    sampler=dict(type='DefaultSampler', shuffle=False),
    dataset=dict(type=dataset_type, metainfo=metainfo,
        data_root=data_root, ann_file='test/_annotations.coco.json',
        data_prefix=dict(img='test/images/'),
        test_mode=True, pipeline=test_pipeline))

val_evaluator = dict(type='CocoMetric',
    ann_file=data_root+'valid/_annotations.coco.json',
    metric='bbox', format_only=False)
test_evaluator = dict(type='CocoMetric',
    ann_file=data_root+'test/_annotations.coco.json',
    metric='bbox', format_only=False)

optim_wrapper = dict(type='OptimWrapper',
    optimizer=dict(type='AdamW', lr=0.0001, weight_decay=0.05),
    paramwise_cfg=dict(custom_keys={{'backbone': dict(lr_mult=0.1)}},
                       norm_decay_mult=0.0))

max_epochs = {MAX_EPOCHS}
train_cfg  = dict(type='EpochBasedTrainLoop', max_epochs=max_epochs, val_interval=1)
val_cfg    = dict(type='ValLoop')
test_cfg   = dict(type='TestLoop')

param_scheduler = [
    dict(type='LinearLR', start_factor=0.001, by_epoch=False, begin=0, end=500),
    dict(type='CosineAnnealingLR', begin=0, end=max_epochs, by_epoch=True,
         T_max=max_epochs, eta_min=1e-6)]

default_scope = 'mmdet'
default_hooks = dict(
    timer=dict(type='IterTimerHook'),
    logger=dict(type='LoggerHook', interval=10),
    param_scheduler=dict(type='ParamSchedulerHook'),
    checkpoint=dict(type='CheckpointHook', interval=5, max_keep_ckpts=3,
        save_best='coco/bbox_mAP', rule='greater'),
    sampler_seed=dict(type='DistSamplerSeedHook'),
    visualization=dict(type='DetVisualizationHook'))

vis_backends = [dict(type='LocalVisBackend'), dict(type='TensorboardVisBackend')]
visualizer   = dict(type='DetLocalVisualizer', vis_backends=vis_backends, name='visualizer')
log_processor= dict(type='LogProcessor', window_size=50, by_epoch=True)
log_level    = 'INFO'
load_from    = None
resume       = False
"""

with open('/content/mmdetection/configs/vision_mamba_ecuaplacas.py', 'w') as f:
    f.write(cfg)
print("✅ Config generado.")

## 7. ✅ Verificación Pre-Entrenamiento

In [ ]:
import os, json
print("Verificando dataset COCO...")
for split in ['train','valid','test']:
    p = f'/content/dataset_coco/{split}/_annotations.coco.json'
    if os.path.exists(p):
        d = json.load(open(p))
        print(f"  ✅ {split:5s}: {len(d['images']):4d} imgs, {len(d['annotations']):5d} anns")
    else:
        print(f"  ❌ {split}: JSON no encontrado!")
ckpt = '/content/mmdetection/checkpoints/vim_tiny_backbone.pth'
sz   = os.path.getsize(ckpt)/(1024**2) if os.path.exists(ckpt) else -1
print(f"  Backbone Vim-Tiny: {sz:.1f} MB" if sz>0 else "  ❌ Backbone no encontrado!")

## 8. 🔥 Entrenamiento

In [ ]:
%cd /content/mmdetection
!python tools/train.py configs/vision_mamba_ecuaplacas.py \
    --work-dir {WORK_DIR} \
    2>&1 | tee {WORK_DIR}/training_log.txt

## 9. 📊 Parseo de Logs → result.csv

MMDetection ≥3.x genera un `scalars.json` en `{work_dir}/{timestamp}/vis_data/`. Esta celda lo parsea y construye un DataFrame por época.

In [ ]:
import os, json, glob, re
import pandas as pd
import numpy as np
from pathlib import Path

def find_scalars_json(work_dir):
    files = glob.glob(os.path.join(work_dir,'*','vis_data','scalars.json'))
    if files: return sorted(files)[-1]
    direct = os.path.join(work_dir,'vis_data','scalars.json')
    return direct if os.path.exists(direct) else None

def parse_mmdet_scalars(work_dir):
    path = find_scalars_json(work_dir)
    if not path:
        print("⚠️  scalars.json no encontrado, parseando log .txt...")
        return parse_log_txt(work_dir)
    print(f"📄 {path}")
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try: records.append(json.loads(line))
                except: pass
    if not records:
        return parse_log_txt(work_dir)
    df = pd.DataFrame(records)

    # Separar filas de train (tienen loss) y val (tienen mAP)
    is_train = df.filter(like='loss').notna().any(axis=1)
    is_val   = df.filter(like='mAP').notna().any(axis=1) | df.filter(like='coco').notna().any(axis=1)

    df_tr = df[is_train].copy()
    df_vl = df[is_val].copy()

    def prefix(df_, pre):
        return df_.rename(columns={c: f'{pre}/{c}' for c in df_.columns if c != 'epoch'})

    if not df_tr.empty and 'epoch' in df_tr.columns:
        agg   = {c:'mean' for c in df_tr.columns if c not in ('epoch','iter','step')}
        df_tr = df_tr.groupby('epoch').agg(agg).reset_index()
        df_tr = prefix(df_tr, 'train')

    if not df_vl.empty:
        df_vl = prefix(df_vl, 'val')

    if not df_tr.empty and not df_vl.empty and 'epoch' in df_vl.columns:
        return pd.merge(df_tr, df_vl, on='epoch', how='outer').sort_values('epoch')
    return df_tr if not df_tr.empty else df_vl

def parse_log_txt(work_dir):
    path = os.path.join(work_dir,'training_log.txt')
    if not os.path.exists(path): return pd.DataFrame()
    print(f"📄 {path}")
    content = open(path, errors='ignore').read()

    ep_data = {}
    pat = re.compile(
        r'Epoch\s*\[(\d+)/\d+\]\s*\[\d+/\d+\].*?'
        r'loss:\s*([\d.]+).*?lr:\s*([\d.e+-]+)',
        re.DOTALL)
    for m in pat.finditer(content):
        ep = int(m.group(1))
        ep_data.setdefault(ep,[]).append({'train/loss':float(m.group(2)),'train/lr':float(m.group(3))})

    train_rows = [{'epoch':ep, **{k:np.mean([r[k] for r in rows])
                   for k in rows[0]}} for ep,rows in sorted(ep_data.items())]

    val_rows = []
    for block in re.split(r'(?=Epoch\(val\))', content):
        em = re.search(r'Epoch\(val\)\s*\[(\d+)\]', block)
        if not em: continue
        rec = {'epoch': int(em.group(1))}
        for key, pat2 in [
            ('val/coco_bbox_mAP',   r'bbox_mAP:\s*([\d.]+)'),
            ('val/coco_bbox_mAP_50',r'bbox_mAP_50:\s*([\d.]+)'),
            ('val/coco_bbox_mAP_75',r'bbox_mAP_75:\s*([\d.]+)'),
            ('val/coco_bbox_mAP_s', r'bbox_mAP_s:\s*([\d.]+)'),
            ('val/coco_bbox_mAP_m', r'bbox_mAP_m:\s*([\d.]+)'),
            ('val/coco_bbox_mAP_l', r'bbox_mAP_l:\s*([\d.]+)'),
        ]:
            mm = re.search(pat2, block)
            if mm: rec[key] = float(mm.group(1))
        if len(rec) > 1: val_rows.append(rec)

    df_t = pd.DataFrame(train_rows)
    df_v = pd.DataFrame(val_rows)
    if not df_t.empty and not df_v.empty:
        return pd.merge(df_t, df_v, on='epoch', how='outer').sort_values('epoch')
    return df_t if not df_t.empty else df_v

print("🔍 Parseando métricas...")
df_results = parse_mmdet_scalars(WORK_DIR)

if df_results.empty:
    print("⚠️  No se encontraron métricas.")
else:
    epoch_col = ['epoch'] if 'epoch' in df_results.columns else []
    tr_cols   = sorted([c for c in df_results.columns if c.startswith('train/')])
    vl_cols   = sorted([c for c in df_results.columns if c.startswith('val/')])
    ot_cols   = [c for c in df_results.columns if c not in epoch_col+tr_cols+vl_cols]
    df_results = df_results[epoch_col+tr_cols+vl_cols+ot_cols]
    print(f"✅ {len(df_results)} épocas × {len(df_results.columns)} columnas")
    display(df_results)

## 10. 💾 Guardar result.csv

In [ ]:
from datetime import datetime
import glob

if not df_results.empty:
    ts  = datetime.now().strftime('%Y%m%d_%H%M%S')
    csv_filename = f'vision_mamba_results_{ts}.csv'

    local_csv = os.path.join(WORK_DIR, 'result.csv')
    df_results.to_csv(local_csv, index=False, float_format='%.6f')
    print(f"💾 Colab  : {local_csv}")

    drive_csv = os.path.join(RESULTS_DIR, csv_filename)
    df_results.to_csv(drive_csv, index=False, float_format='%.6f')
    print(f"💾 Drive  : {drive_csv}")

    # Copiar mejor checkpoint al Drive
    bests = sorted(glob.glob(os.path.join(WORK_DIR,'best_coco_bbox_mAP*.pth')))
    if bests:
        dst = os.path.join(RESULTS_DIR, f'vision_mamba_best_{ts}.pth')
        os.system(f'cp "{bests[-1]}" "{dst}"')
        print(f"🏆 Best ckpt: {dst}")

    # Mostrar resumen
    print("\n📊 Últimas métricas:")
    last = df_results.iloc[-1]
    for c in [x for x in df_results.columns if 'mAP' in x or x=='train/loss']:
        v = last.get(c, float('nan'))
        if not pd.isna(v): print(f"   {c:<35s}: {v:.4f}")
else:
    print("⚠️  No hay datos para exportar.")

## 11. 📈 Curvas de Entrenamiento

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

if not df_results.empty:
    fig = plt.figure(figsize=(18, 12))
    fig.suptitle('Vision Mamba · Detección de Placas Ecuatorianas\nCurvas de Entrenamiento',
                 fontsize=16, fontweight='bold', y=1.01)
    gs  = gridspec.GridSpec(2, 3, hspace=0.42, wspace=0.35)

    ep  = df_results['epoch'].values if 'epoch' in df_results.columns else range(len(df_results))

    # 1 - Loss total
    ax = fig.add_subplot(gs[0,0])
    if 'train/loss' in df_results.columns:
        ax.plot(ep, df_results['train/loss'], 'b-o', ms=4)
    ax.set_title('Pérdida Total (Train)',fw='bold'); ax.set_xlabel('Época'); ax.set_ylabel('Loss'); ax.grid(alpha=.3)

    # 2 - Losses detalladas
    ax2 = fig.add_subplot(gs[0,1])
    colors = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6']
    det_cols = [c for c in df_results.columns if c.startswith('train/loss_')]
    for i,c in enumerate(det_cols[:5]):
        ax2.plot(ep, df_results[c], '-o', ms=3, color=colors[i], label=c.replace('train/loss_',''))
    ax2.set_title('Pérdidas Detalladas',fw='bold'); ax2.set_xlabel('Época'); ax2.legend(fs=8); ax2.grid(alpha=.3)

    # 3 - LR
    ax3 = fig.add_subplot(gs[0,2])
    if 'train/lr' in df_results.columns:
        ax3.semilogy(ep, df_results['train/lr'], 'g-o', ms=4)
    ax3.set_title('Learning Rate',fw='bold'); ax3.set_xlabel('Época'); ax3.set_ylabel('LR'); ax3.grid(alpha=.3)

    # 4 - mAP general
    ax4 = fig.add_subplot(gs[1,0])
    for c in [x for x in df_results.columns if 'mAP' in x and 'val' in x][:3]:
        ok = df_results[c].notna()
        ax4.plot(df_results['epoch'][ok] if 'epoch' in df_results.columns else range(ok.sum()),
                 df_results[c][ok], '-s', ms=5, label=c.split('_')[-1])
    ax4.set_ylim([0,1]); ax4.set_title('mAP (Validación)',fw='bold'); ax4.legend(); ax4.grid(alpha=.3)

    # 5 - mAP@50 vs mAP@75
    ax5 = fig.add_subplot(gs[1,1])
    for c,col,ls in [('val/coco_bbox_mAP_50','#e74c3c','-'),('val/coco_bbox_mAP_75','#3498db','--')]:
        if c in df_results.columns:
            ok = df_results[c].notna()
            ax5.plot(df_results['epoch'][ok], df_results[c][ok], ls+'s', ms=5, color=col, label=c[-2:])
    ax5.set_ylim([0,1]); ax5.set_title('mAP@50 vs mAP@75',fw='bold'); ax5.legend(); ax5.grid(alpha=.3)

    # 6 - Tabla resumen
    ax6 = fig.add_subplot(gs[1,2]); ax6.axis('off')
    rows_tbl = []
    for c in [x for x in df_results.columns if 'mAP' in x or x=='train/loss']:
        v = df_results[c].dropna()
        if v.empty: continue
        best = v.max() if 'mAP' in c else v.min()
        idx  = v.idxmax() if 'mAP' in c else v.idxmin()
        ep_b = df_results['epoch'].iloc[idx] if 'epoch' in df_results.columns else '-'
        rows_tbl.append([c.replace('val/coco_bbox_','').replace('train/',''), f'{best:.4f}', str(int(ep_b))])
    if rows_tbl:
        tbl = ax6.table(cellText=rows_tbl, colLabels=['Métrica','Mejor','Época'],
                        cellLoc='center', loc='center')
        tbl.auto_set_font_size(True); tbl.scale(1,1.5)
    ax6.set_title('Resumen Mejores Métricas', fw='bold', pad=20)

    plt.tight_layout()
    plot_path = os.path.join(WORK_DIR, 'training_curves.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    os.system(f'cp "{plot_path}" "{os.path.join(RESULTS_DIR, f"training_curves_{ts}.png")}"')
    print(f"📊 Gráfica: {plot_path}")
    plt.show()
else:
    print("⚠️  No hay datos para graficar.")

## 12. 🧪 Evaluación en Test Set

In [ ]:
import glob
bests = sorted(glob.glob(os.path.join(WORK_DIR,'best_coco_bbox_mAP*.pth')))
if not bests:
    bests = sorted(glob.glob(os.path.join(WORK_DIR,'epoch_*.pth')))
best_ckpt = bests[-1] if bests else None

if best_ckpt:
    print(f"🏆 Evaluando: {best_ckpt}")
    %cd /content/mmdetection
    !python tools/test.py configs/vision_mamba_ecuaplacas.py \
        "{best_ckpt}" --work-dir {WORK_DIR} \
        2>&1 | tee {WORK_DIR}/test_results.txt
    print("\n✅ Test completado.")
else:
    print("❌ No hay checkpoint disponible.")

## 13. 📋 Añadir Test Metrics al CSV

In [ ]:
import re, pd as pandas_alias

def parse_test_log(path):
    if not os.path.exists(path): return {}
    content = open(path, errors='ignore').read()
    out = {}
    for key, pat in [
        ('test/coco_bbox_mAP',    r'bbox_mAP:\s*([\d.]+)'),
        ('test/coco_bbox_mAP_50', r'bbox_mAP_50:\s*([\d.]+)'),
        ('test/coco_bbox_mAP_75', r'bbox_mAP_75:\s*([\d.]+)'),
        ('test/coco_bbox_mAP_s',  r'bbox_mAP_s:\s*([\d.]+)'),
        ('test/coco_bbox_mAP_m',  r'bbox_mAP_m:\s*([\d.]+)'),
        ('test/coco_bbox_mAP_l',  r'bbox_mAP_l:\s*([\d.]+)'),
    ]:
        m = re.search(pat, content)
        if m: out[key] = float(m.group(1))
    return out

test_metrics = parse_test_log(os.path.join(WORK_DIR,'test_results.txt'))
print("🧪 Test Set Metrics:")
for k,v in test_metrics.items(): print(f"   {k:<35s}: {v:.4f}")

if test_metrics and not df_results.empty:
    import pandas as pd
    test_row = pd.DataFrame([{'epoch':'TEST', **test_metrics}])
    df_final = pd.concat([df_results, test_row], ignore_index=True)
    df_final.to_csv(os.path.join(WORK_DIR,'result.csv'), index=False, float_format='%.6f')
    df_final.to_csv(os.path.join(RESULTS_DIR,csv_filename), index=False, float_format='%.6f')
    print("\n✅ result.csv actualizado con fila TEST.")
    display(test_row)

## 14. ⬇️ Descarga de Archivos

In [ ]:
from google.colab import files
for fp in [os.path.join(WORK_DIR,'result.csv'),
           os.path.join(WORK_DIR,'training_curves.png'),
           os.path.join(WORK_DIR,'test_results.txt')]:
    if os.path.exists(fp):
        files.download(fp)
        print(f"⬇️  {os.path.basename(fp)}")
    else:
        print(f"⚠️  No encontrado: {fp}")